# Stage 3: Direct Preference Optimization (DPO) Alignment

Direct Preference Optimization (DPO) aligns the language model with human feedback by utilizing preference datasets. Each sample has a prompt, a chosen (preferred, high quality, helpful) response, and a rejected (dismissive, poor, unhelpful) response. DPO trains the model to maximize the log-likelihood of the chosen response while minimizing the rejected response, refining its tone and safety.

### Step 1: Install Required Libraries

In [ ]:
# Install Unsloth and standard ML libraries
# Uncomment the lines below to install if running in a GPU cloud env (like Colab)
#!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
#!pip install --no-deps "xformers" "trl" peft accelerate bitsandbytes

### Step 2: Load SFT Model and Tokenizer
We load the Supervised Fine-Tuning (SFT) adapter weights from Stage 2 as our starting point.

In [ ]:
from unsloth import FastLanguageModel
import torch
import os

max_seq_length = 2048
dtype = None
load_in_4bit = True

# Option A: Load from local SFT adapters
adapters_dir = "adapters" if os.path.exists("data") else "../adapters"
sft_adapters_path = os.path.join(adapters_dir, "sft_lora")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = sft_adapters_path,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Option B: Load from SFT adapters saved on Hugging Face (Uncomment to use)
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "techsivam/sft_lora",
#     max_seq_length = max_seq_length,
#     dtype = dtype,
#     load_in_4bit = load_in_4bit,
# )

# Reapply LoRA for DPO training
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### Step 3: Load and Format Preference Dataset
We load `data/preference_dataset.jsonl` and format it with prompt, chosen, and rejected templates.

In [ ]:
import os
import json
from datasets import Dataset

# Check path compatibility (Colab vs Local)
data_path = "data/preference_dataset.jsonl" if os.path.exists("data/preference_dataset.jsonl") else "../data/preference_dataset.jsonl"

pref_data = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            pref_data.append(json.loads(line))

# DPO requires 'prompt', 'chosen', and 'rejected'
# We must apply standard prompt format
formatted_prompts = []
formatted_chosen = []
formatted_rejected = []

system_prompt = "<|im_start|>system\nYou are a professional IT Helpdesk Assistant. Answer the user's technical questions accurately and professionally.<|im_end|>\n"

for item in pref_data:
    # Prompt input
    prompt_text = system_prompt + f"<|im_start|>user\n{item['prompt']}<|im_end|>\n<|im_start|>assistant\n"
    formatted_prompts.append(prompt_text)
    
    # Chosen vs Rejected answers
    formatted_chosen.append(item['chosen'] + tokenizer.eos_token)
    formatted_rejected.append(item['rejected'] + tokenizer.eos_token)

dataset = Dataset.from_dict({
    "prompt": formatted_prompts,
    "chosen": formatted_chosen,
    "rejected": formatted_rejected
})
print(f"Loaded and formatted {len(dataset)} preference examples for DPO.")

### Step 4: Configure and Run DPOTrainer
We use DPOTrainer from the TRL library with optimized settings.

In [ ]:
import inspect
from trl import DPOTrainer
from transformers import TrainingArguments
from unsloth import PatchDPOTrainer

# Unsloth patches DPOTrainer to make it 2x faster
PatchDPOTrainer()

# Try to import DPOConfig for newer TRL versions
try:
    from trl import DPOConfig
except ImportError:
    DPOConfig = None

if DPOConfig is not None:
    # Newer TRL config style (v0.8.0+)
    dpo_args = DPOConfig(
        beta = 0.1,
        max_prompt_length = 1024,
        max_length = 2048,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 40, # Number of steps for alignment
        learning_rate = 5e-6, # Lower LR for DPO to avoid disruption
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs_dpo",
    )
else:
    # Older TRL style
    dpo_args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 40, # Number of steps for alignment
        learning_rate = 5e-6, # Lower LR for DPO to avoid disruption
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs_dpo",
    )

# Prepare DPOTrainer arguments
sig_dpo = inspect.signature(DPOTrainer.__init__)
dpo_kwargs = {
    "model": model,
    "ref_model": None,
    "train_dataset": dataset,
    "args": dpo_args,
}

if DPOConfig is None:
    # If DPOConfig is None, these must be passed directly to DPOTrainer
    dpo_kwargs["beta"] = 0.1
    dpo_kwargs["max_prompt_length"] = 1024
    dpo_kwargs["max_length"] = 2048

if "processing_class" in sig_dpo.parameters:
    dpo_kwargs["processing_class"] = tokenizer
else:
    dpo_kwargs["tokenizer"] = tokenizer

dpo_trainer = DPOTrainer(**dpo_kwargs)
dpo_trainer.train()

### Step 5: Test DPO-Aligned Model
Verify DPO performance on IT questions.

In [ ]:
FastLanguageModel.for_inference(model)

eval_prompt = system_prompt + "<|im_start|>user\nSubject: VPN issue\nQuery: Cannot connect to VPN from home.<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer([eval_prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

### Step 6: Save Aligned Adapters

In [ ]:
import os
# Check path compatibility (Colab/Root vs Local)
adapters_dir = "adapters" if os.path.exists("data") else "../adapters"
save_path = os.path.join(adapters_dir, "dpo_lora")

# Option A: Save locally
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Stage 3 (DPO) LoRA adapters saved successfully to {save_path}!")

# Option B: Push to Hugging Face Hub (Uncomment to use)
# model.push_to_hub("techsivam/dpo_lora", private=True)
# tokenizer.push_to_hub("techsivam/dpo_lora", private=True)